══════════════════════════════════════════════════════════════
 WEEK 10  |  DAY 5  |  PROJECT — BUILD A RAG PIPELINE

══════════════════════════════════════════════════════════════

 LEARNING OBJECTIVES
 ───────────────────
 This project combines everything from Week 10:
 1. Understand what RAG (Retrieval-Augmented Generation) is
 2. Build a complete RAG pipeline from scratch
 3. Connect ChromaDB + LLM to answer questions from a custom knowledge base
 4. Understand the limitations and when to use RAG vs fine-tuning

 TIME:  ~50 minutes

 YOUTUBE
 ───────
 Search: "RAG pipeline Python from scratch tutorial"
 Search: "LangChain RAG tutorial ChromaDB 2025"
 Search: "RAG vs fine-tuning when to use each"

 INSTALL:
   pip install chromadb openai langchain langchain-openai

 ─────────────────────────────────────────────────────────────
 ARCHITECTURE DECISION
 ─────────────────────
 Choosing between: naive RAG (chunk + embed + retrieve) vs hybrid retrieval (BM25 + vector) vs agentic RAG.
 Rule of thumb: start with naive RAG. Add BM25 hybrid when keyword matches matter (exact codes, product names). Move to agentic RAG only when a single retrieval step consistently fails to find the right context.

══════════════════════════════════════════════════════════════

In [ ]:
import os
import json

══════════════════════════════════════════════════════════════
 CONCEPT 1 — WHAT IS RAG?

══════════════════════════════════════════════════════════════

RAG = Retrieval-Augmented Generation

Problem: LLMs have a knowledge cutoff (they don't know your company's data).
Solution: Before asking the LLM a question, RETRIEVE relevant documents
          from your own knowledge base, then AUGMENT the prompt with them.

RAG Pipeline:
  1. INDEX:   Load documents -> chunk them -> embed -> store in vector DB
  2. QUERY:   User asks a question -> embed the question -> find similar chunks
  3. GENERATE: Send question + retrieved chunks to LLM -> get answer

Simple diagram:
  User question
      |
  Embed question  -> Vector DB -> Find top-K similar chunks
                                       |
  Build prompt: "Answer this question using ONLY the context below:
                 Context: [retrieved chunks]
                 Question: [user question]"
                                       |
                                LLM -> Final answer

EXAMPLE ──────────────────────────────────────────────────────

In [ ]:
print("=" * 55)
print("PROJECT: RAG Pipeline")
print("=" * 55)
print()
print("We will build a Q&A system over a custom knowledge base.")
print("The system will only answer from the provided documents.")

══════════════════════════════════════════════════════════════
 CONCEPT 2 — PREPARE THE KNOWLEDGE BASE

══════════════════════════════════════════════════════════════

EXAMPLE ──────────────────────────────────────────────────────

In [ ]:
print()
print("STEP 1: Knowledge Base")
print("-" * 40)

# Our knowledge base: data engineering concepts
# In a real project these could come from PDFs, docs, confluence pages, etc.
knowledge_base = [
    {
        "id": "etl_001",
        "text": "ETL stands for Extract, Transform, Load. It is the process of moving "
                "data from source systems, cleaning and reshaping it, and loading it "
                "into a destination like a data warehouse. Common ETL tools include "
                "Apache Spark, dbt, Talend, and Python with pandas."
    },
    {
        "id": "dw_001",
        "text": "A data warehouse is a centralized repository for structured data "
                "optimized for analytical queries. Unlike transactional databases (OLTP), "
                "data warehouses (OLAP) are optimized for read-heavy workloads. "
                "Popular options: Snowflake, BigQuery, Redshift, Azure Synapse."
    },
    {
        "id": "airflow_001",
        "text": "Apache Airflow is an open-source workflow orchestration tool. "
                "Pipelines are defined as DAGs (Directed Acyclic Graphs) in Python. "
                "Each node in the DAG is a Task. Airflow schedules, monitors, and "
                "retries tasks automatically. Alternatives include Prefect and Luigi."
    },
    {
        "id": "spark_001",
        "text": "Apache Spark is a distributed computing framework for processing "
                "large datasets. PySpark is the Python API for Spark. Spark stores "
                "data in RDDs and DataFrames, and processes data in memory for speed. "
                "Use Spark when data exceeds what fits on a single machine (typically > 1TB)."
    },
    {
        "id": "sql_join_001",
        "text": "SQL JOINs combine rows from two or more tables based on a related column. "
                "INNER JOIN returns only matching rows. LEFT JOIN returns all left-table rows "
                "plus matching right-table rows (nulls where no match). "
                "RIGHT JOIN is the reverse. FULL OUTER JOIN returns all rows from both tables."
    },
    {
        "id": "pandas_001",
        "text": "Pandas is a Python library for data manipulation and analysis. "
                "The main data structure is a DataFrame — a 2D table with labeled columns. "
                "Key operations: read_csv, read_excel, merge, groupby, pivot_table, fillna. "
                "Pandas is best for datasets that fit in memory (typically < 1-2GB)."
    },
    {
        "id": "api_001",
        "text": "A REST API (Application Programming Interface) allows systems to "
                "communicate over HTTP. GET requests retrieve data, POST requests send data. "
                "Responses are typically in JSON format. In Python, use the 'requests' library. "
                "Always handle errors: check response.status_code before using the data."
    },
    {
        "id": "dbt_001",
        "text": "dbt (data build tool) is a transformation tool for analytics engineering. "
                "It runs SQL SELECT statements and creates tables/views in your data warehouse. "
                "dbt handles dependency management, testing, and documentation automatically. "
                "It follows ELT (Extract, Load, Transform) rather than ETL."
    },
]

print(f"Knowledge base: {len(knowledge_base)} documents loaded")
for doc in knowledge_base:
    print(f"  [{doc['id']}] {doc['text'][:60]}...")

══════════════════════════════════════════════════════════════
 CONCEPT 3 — RETRIEVE AND GENERATE

══════════════════════════════════════════════════════════════

The retrieval step finds the most relevant documents for a question.
The generation step sends the question + context to the LLM.

EXAMPLE ──────────────────────────────────────────────────────

In [ ]:
print()
print("STEP 2: Index Documents in ChromaDB")
print("-" * 40)

# import chromadb
#
# chroma_client = chromadb.Client()
# collection = chroma_client.create_collection("de_knowledge_base")
#
# collection.add(
#     documents=[doc["text"] for doc in knowledge_base],
#     ids=[doc["id"] for doc in knowledge_base]
# )
# print(f"Indexed {len(knowledge_base)} documents in ChromaDB")

print("(Uncomment ChromaDB code after: pip install chromadb)")

print()
print("STEP 3: Retrieval Function")
print("-" * 40)

def retrieve(question, n_results=2):
    """
    Find the most relevant documents for a question.
    Returns a list of text chunks.
    """
    # Uncomment once ChromaDB is set up:
    # results = collection.query(query_texts=[question], n_results=n_results)
    # return results["documents"][0]

    # Placeholder — simple keyword search for now
    question_lower = question.lower()
    matches = []
    for doc in knowledge_base:
        if any(word in doc["text"].lower() for word in question_lower.split()):
            matches.append(doc["text"])
        if len(matches) >= n_results:
            break
    return matches if matches else [knowledge_base[0]["text"]]

# Test the retrieval
test_questions = [
    "What is Airflow used for?",
    "When should I use Spark instead of pandas?",
    "What is the difference between ETL and ELT?",
]

for q in test_questions:
    chunks = retrieve(q)
    print(f"\nQ: {q}")
    print(f"  Retrieved: {chunks[0][:80]}...")

print()
print("STEP 4: Generate Answers with LLM")
print("-" * 40)

def answer_question(question):
    """
    RAG pipeline: retrieve relevant docs -> build augmented prompt -> call LLM.
    """
    # Step 1: Retrieve
    context_chunks = retrieve(question, n_results=2)
    context = "\n\n".join(context_chunks)

    # Step 2: Build augmented prompt
    system_prompt = (
        "You are a data engineering assistant. "
        "Answer ONLY using the context provided. "
        "If the answer is not in the context, say 'I don't have that information.'"
    )
    user_prompt = f"Context:\n{context}\n\nQuestion: {question}"

    # Step 3: Call LLM (uncomment once you have an API key)
    # from openai import OpenAI
    # client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))
    # response = client.chat.completions.create(
    #     model="gpt-4o-mini",
    #     messages=[
    #         {"role": "system", "content": system_prompt},
    #         {"role": "user",   "content": user_prompt}
    #     ],
    #     max_tokens=200
    # )
    # return response.choices[0].message.content.strip()

    # Placeholder response:
    return f"[LLM would answer using context: {context[:100]}...]"

# Test the full pipeline
print()
for question in test_questions:
    print(f"Q: {question}")
    answer = answer_question(question)
    print(f"A: {answer}")
    print()

══════════════════════════════════════════════════════════════
 EXERCISE 1

══════════════════════════════════════════════════════════════
Expand the knowledge base by adding 3 more documents covering
topics you find important.
Ideas: Python virtual environments, git for data engineers, Docker basics,
       data quality checks, streaming vs batch processing.
Re-index after adding (re-run the ChromaDB step).

Expected output:
    knowledge_base list now has 11 documents
    ChromaDB collection has 11 indexed documents

══════════════════════════════════════════════════════════════
 EXERCISE 2

══════════════════════════════════════════════════════════════
Ask the system a question it CANNOT answer from the knowledge base:
  "What is the capital of France?"

The LLM should respond: "I don't have that information."
Test this and verify the system doesn't hallucinate.
Print the question and the answer.

Expected output:
    Q: What is the capital of France?
    A: I don't have that information.

══════════════════════════════════════════════════════════════
 EXERCISE 3

══════════════════════════════════════════════════════════════
Modify the answer_question() function to also return the SOURCE document IDs
that were used. Print the answer AND the sources.

Expected output:
    Q: What is Airflow used for?
    A: Apache Airflow is a workflow orchestration tool...
    Sources: airflow_001

══════════════════════════════════════════════════════════════
 FINAL REFLECTION

══════════════════════════════════════════════════════════════
Answer these questions as comments:

  1. What is the main advantage of RAG over asking the LLM directly?
  2. What happens if the knowledge base has outdated or wrong information?
  3. When would you fine-tune a model instead of using RAG?
     (Hint: fine-tuning changes the model itself; RAG adds external knowledge)

══════════════════════════════════════════════════════════════
 CONCEPT 4 — FINE-TUNING vs RAG vs PROMPTING
══════════════════════════════════════════════════════════════

When RAG isn't enough, you have two more options: better prompting
and fine-tuning. Choosing the right tool saves time and money.

BETTER PROMPTING — try this first, it costs nothing
  What it changes: how the model interprets and responds to your request
  Good for: fixing output format, adding persona, basic task adjustments
  Not good for: adding knowledge the model doesn't have

RAG — use when the problem is knowledge
  What it changes: the context injected at query time (model unchanged)
  Good for: large/changing knowledge bases, needing to cite sources
  Not good for: changing model style, behavior, or format

FINE-TUNING — use when prompting and RAG both fail
  What it changes: the model's weights (permanent behavior change)
  Good for: specific writing style, consistent output format, domain jargon
  Requires: 100+ labeled examples, training time, training cost
  Warning: fine-tuning does NOT reliably add factual knowledge —
           the model can still hallucinate even after fine-tuning on facts

DECISION FLOW:
  Is the problem about KNOWLEDGE or FACTS?
    Yes → RAG
  Is the problem about FORMAT, STYLE, or TONE?
    Try prompt engineering first → if still failing → fine-tuning
  Does the model need to "know" your internal data?
    → RAG (for retrieval) OR fine-tuning (for style, not facts)

EXAMPLE ──────────────────────────────────────────────────────

In [ ]:

print()
print("=" * 55)
print("CONCEPT 4: Fine-Tuning vs RAG vs Prompting")
print("=" * 55)
print()
print("THREE OPTIONS for improving LLM behavior:")
print()
print("  BETTER PROMPTING (try this first — free, instant)")
print("    Improve the system prompt, add few-shot examples")
print("    Use for: output format, tone, simple task adjustments")
print("    Limit: can't add new knowledge, model may still deviate")
print()
print("  RAG (retrieval-augmented generation)")
print("    Keep model unchanged, inject relevant documents at query time")
print("    Use for: large/changing knowledge bases, citing sources")
print("    Limit: only as good as your retrieval; adds latency")
print()
print("  FINE-TUNING (retrain model on your data)")
print("    Train the base model on your labeled examples")
print("    Use for: specific style/tone, new output format, niche domain")
print("    Limit: expensive, needs 100s+ examples, can cause hallucination")
print()
print("DECISION RULES:")
print("  Problem is about KNOWLEDGE  → RAG")
print("  Problem is about STYLE/FORMAT → fine-tuning or prompting")
print("  Try prompting first — it's free and often enough")
print("  Fine-tune only when prompting + RAG both fail")
print()
print("COMMON MISTAKE: fine-tuning to add facts.")
print("  Fine-tuning does NOT reliably add factual knowledge.")
print("  The model may still hallucinate. Use RAG for facts.")


══════════════════════════════════════════════════════════════
 EXERCISE 4 — Choose: RAG, Fine-tuning, or Prompting?
══════════════════════════════════════════════════════════════
For each scenario in the list below, write as a comment:
  1. Your choice: RAG / fine-tuning / better prompting
  2. One-sentence reason

Expected output (as comments):
    Scenario 1 (formal English style):
      Choice: fine-tuning
      Reason: style is a model behavior change, not a knowledge problem

    Scenario 2 (search 50k product descriptions):
      Choice: RAG
      Reason: large knowledge base that needs semantic search

    Scenario 3 (200 contract documents):
      Choice: RAG
      Reason: needs to retrieve specific contract clauses on demand

    Scenario 4 (always output SQL):
      Choice: better prompting (system prompt + few-shot)
      Reason: output format can be enforced with a clear prompt and examples

    Scenario 5 (founder writing style):
      Choice: fine-tuning
      Reason: style/voice is a behavior pattern best learned from examples

In [ ]:

scenarios = [
    "A company wants their support chatbot to always respond in formal, legal-style English.",
    "A company has 50,000 product descriptions and wants customers to search by meaning.",
    "A legal firm wants staff to ask questions about 200 internal contract documents.",
    "A startup wants their AI to always output answers in valid SQL, never prose.",
    "A company wants their AI to respond exactly like their founder writes.",
]






══════════════════════════════════════════════════════════════
 CONCEPT 5 — VERTEX AI ORCHESTRATION: WHEN TO GO MANAGED
══════════════════════════════════════════════════════════════
 You have built RAG pipelines with LangChain running locally.
 For production, you must choose: local/custom vs managed cloud.

 OPTION A — LangChain (local / self-managed)
   + Full control, low cost for small teams
   + Fast iteration, no cloud setup
   - You own reliability, scaling, monitoring
   - No built-in audit trail or compliance artifacts

 OPTION B — Vertex AI Pipelines (Google Cloud managed)
   + Each step is a containerized component — fully auditable
   + Built-in lineage tracking, IAM, and SOC2-compliant logging
   + Scales automatically, managed retries
   - Higher cost, vendor lock-in
   - Requires Google Cloud project, IAM setup

 OPTION C — Airflow / Prefect (self-hosted orchestration)
   + Mature DAG scheduling, retries, alerting
   + Works with any cloud or on-premise
   - Operational overhead of running the orchestrator itself

 Architecture decision rule:
   Prototype / startup → LangChain local
   Team > 5 / compliance required → Vertex AI or managed Airflow
   Existing Airflow infra → extend it, don't replace it

 EXAMPLE ──────────────────────────────────────────────────────

In [ ]:
# SIMULATION MODE — shows what a Vertex AI pipeline step looks like locally.
# REAL MODE (commented below) requires: pip install google-cloud-aiplatform

from dataclasses import dataclass
from typing import Callable
import time


@dataclass
class PipelineStep:
    """Represents one step in a Vertex AI pipeline (simulated locally)."""
    name: str
    fn: Callable
    inputs: dict

    def run(self) -> dict:
        print(f"  [{self.name}] starting...")
        start = time.time()
        result = self.fn(**self.inputs)
        elapsed = time.time() - start
        print(f"  [{self.name}] done in {elapsed:.2f}s → {list(result.keys())}")
        return result


class LocalPipeline:
    """Runs steps in sequence, passing outputs to next step inputs."""
    def __init__(self, name: str):
        self.name = name
        self.steps = []

    def add_step(self, step: PipelineStep):
        self.steps.append(step)

    def run(self):
        print(f"Pipeline: {self.name}")
        context = {}
        for step in self.steps:
            merged_inputs = {**step.inputs, **context}
            step.inputs = merged_inputs
            output = step.run()
            context.update(output)
        print(f"Pipeline complete. Final context keys: {list(context.keys())}")
        return context


# Define pipeline steps (SIMULATION MODE)
def ingest_documents(source_path: str) -> dict:
    time.sleep(0.05)
    return {"documents": [f"doc_{i}" for i in range(10)], "count": 10}

def embed_documents(documents: list, model: str = "text-embedding-3-small") -> dict:
    time.sleep(0.1)
    return {"embeddings": [[0.1] * 5 for _ in documents], "model": model}

def store_in_vector_db(embeddings: list, documents: list) -> dict:
    time.sleep(0.05)
    return {"index_size": len(embeddings), "status": "indexed"}


pipeline = LocalPipeline("rag_ingestion_v2")
pipeline.add_step(PipelineStep("ingest",  ingest_documents, {"source_path": "/data/contracts"}))
pipeline.add_step(PipelineStep("embed",   embed_documents,  {"model": "text-embedding-3-small"}))
pipeline.add_step(PipelineStep("store",   store_in_vector_db, {}))
result = pipeline.run()

# REAL MODE — Vertex AI (requires Google Cloud project)
# from google.cloud import aiplatform
# aiplatform.init(project="my-project", location="us-central1")
# job = aiplatform.PipelineJob(display_name="rag-pipeline", template_path="rag_pipeline.yaml")
# job.run()

══════════════════════════════════════════════════════════════
 EXERCISE 5 — ORCHESTRATION ARCHITECTURE DECISION
══════════════════════════════════════════════════════════════
 Part A — Architecture Decision:
 For each scenario below, write a comment stating which option you would
 choose (LangChain local / Vertex AI / Airflow) and one-sentence reason.

 Part B — Local Pipeline:
 Build a LocalPipeline for the document processing use case using
 the pipeline_steps functions below. Chain them in order:
   validate_documents → chunk_text → generate_embeddings → save_index

 Print the final pipeline result.

 Expected output (Part B):
     Pipeline: document_processing
       [validate_documents] starting...
       [validate_documents] done in X.XXs → ['valid_docs', 'rejected']
       [chunk_text] starting...
       [chunk_text] done in X.XXs → ['chunks', 'chunk_count']
       [generate_embeddings] starting...
       [generate_embeddings] done in X.XXs → ['embeddings', 'model_used']
       [save_index] starting...
       [save_index] done in X.XXs → ['index_id', 'vectors_stored']
     Pipeline complete.

 --- starting data ---

In [ ]:
import time
from dataclasses import dataclass
from typing import Callable

# Part A — write your architecture decision as comments:
# Scenario 1: 2-person startup, building first RAG chatbot, no compliance requirements
# Choice: ...  Reason: ...

# Scenario 2: Bank, 20 data scientists, SOC2 + GDPR required, existing GCP contract
# Choice: ...  Reason: ...

# Scenario 3: Mid-size SaaS company, Airflow already running for ETL, adding ML pipeline
# Choice: ...  Reason: ...

# Part B — pipeline step functions:
def validate_documents(source_path: str) -> dict:
    time.sleep(0.03)
    return {"valid_docs": ["doc_1", "doc_2", "doc_3"], "rejected": []}

def chunk_text(valid_docs: list, chunk_size: int = 512) -> dict:
    time.sleep(0.05)
    return {"chunks": [f"chunk_{i}" for i in range(len(valid_docs) * 3)], "chunk_count": len(valid_docs) * 3}

def generate_embeddings(chunks: list) -> dict:
    time.sleep(0.08)
    return {"embeddings": [[0.1] * 768 for _ in chunks], "model_used": "text-embedding-3-small"}

def save_index(embeddings: list, chunks: list) -> dict:
    time.sleep(0.03)
    return {"index_id": "idx_20250101_001", "vectors_stored": len(embeddings)}